# 🌙 Baseline 2 — 가우시안 JDC (Joint Denoising + Compression)

**목적 (Notion 연구 계획서 기준):**
> Baseline 2 (가우시안 JDC): 밝은 사진에 `torch.randn()`으로 가짜 노이즈를 씌워 학습시킨 모델
> → **기존 논문 방식(인공 가우시안 노이즈)은 실제 야간 환경(Domain Shift)에서 실패함을 증명**

- **Baseline 1** : 순정 CompressAI (저조도 뭉개짐 확인)
- **Baseline 2 (본 노트북)** : 깨끗한 사진 + 가우시안 노이즈 → 노이즈 제거 + 압축을 동시에 학습 (JDC)
- 학습은 "인공 노이즈"로 하지만, 마지막에 **진짜 LOL 저조도 사진**을 넣어 → 밝기 개선이 안 되고 실패하는 모습을 보여줌 (Domain Shift 증명)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# compressai 설치 (numpy/scipy 버전 충돌 방지를 위해 버전 고정)
!pip install compressai "numpy==1.26.4" "scipy==1.13.1"
# ⚠️ 설치 후 반드시 [런타임 → 세션 다시 시작]! 이후 이 셀은 건너뛰고 아래부터 실행

In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import numpy as np
import math
import torch.nn as nn
import torch.optim as optim
from compressai.zoo import bmshj2018_hyperprior

In [ ]:
class GaussianNoiseDataset(Dataset):
    """깨끗한 사진(high)에 가우시안 노이즈를 인공적으로 씌우는 데이터셋.
    - Input : 깨끗한 사진 + torch.randn() 노이즈 (가짜 노이즈)
    - Target: 깨끗한 사진 (정답)
    ※ Baseline 1과 달리 'low' 폴더는 쓰지 않습니다. (인공 노이즈로만 학습)
    """
    def __init__(self, root_dir, crop_size=256, noise_sigma=25/255.0):
        self.high_dir = os.path.join(root_dir, 'high')
        self.image_names = sorted(os.listdir(self.high_dir))
        self.crop_size = crop_size
        self.noise_sigma = noise_sigma  # 노이즈 세기 (0~1 스케일). 25/255 ≈ 표준적인 세기

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        # 1. 깨끗한 사진 불러오기
        img_name = self.image_names[idx]
        clean_img = Image.open(os.path.join(self.high_dir, img_name)).convert('RGB')

        # 2. 256x256 무작위 크롭
        w, h = TF.get_image_size(clean_img)
        crop_i = random.randint(0, h - self.crop_size)
        crop_j = random.randint(0, w - self.crop_size)
        clean_img = TF.crop(clean_img, crop_i, crop_j, self.crop_size, self.crop_size)

        # 3. 50% 확률 좌우 반전 (Data Augmentation)
        if random.random() > 0.5:
            clean_img = TF.hflip(clean_img)

        # 4. 텐서(0~1)로 변환 -> 이게 '정답(clean)'
        clean_tensor = TF.to_tensor(clean_img)

        # 5. ★ 핵심: 정답에 가우시안 노이즈를 인공으로 씌워서 '입력(noisy)' 생성
        noise = torch.randn_like(clean_tensor) * self.noise_sigma
        noisy_tensor = torch.clamp(clean_tensor + noise, 0.0, 1.0)

        return noisy_tensor, clean_tensor

print("✅ 가우시안 노이즈(JDC) 데이터셋 클래스 준비 완료!")

In [ ]:
# our485 폴더의 'high'(깨끗한 사진)만 사용해서 인공 노이즈를 씌웁니다.
dataset_path = '/content/drive/MyDrive/CUAI_summer_conference/LOL_Dataset/lol_dataset/our485'

train_dataset = GaussianNoiseDataset(root_dir=dataset_path, crop_size=256, noise_sigma=25/255.0)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# 딱 한 배치만 뽑아서 확인
noisy_batch, clean_batch = next(iter(train_loader))
print(f"📦 노이즈(입력) 텐서 크기: {noisy_batch.shape}")
print(f"📦 깨끗한(정답) 텐서 크기: {clean_batch.shape}")

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for i in range(4):
    axes[0, i].imshow(noisy_batch[i].permute(1, 2, 0))
    axes[0, i].set_title(f"Noisy Input {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(clean_batch[i].permute(1, 2, 0))
    axes[1, i].set_title(f"Clean (Ground Truth) {i+1}")
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print("🚀 1. 학습 준비 중... (가우시안 JDC)")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ 사용 중인 장치: {device}")

# 1. 모델 불러오기 (Baseline 1과 동일한 뼈대)
model = bmshj2018_hyperprior(quality=2, pretrained=True).to(device)
model.train()

# 2. 옵티마이저 2개 (본 가중치 / 엔트로피 quantiles)
parameters = [p for n, p in model.named_parameters() if not n.endswith(".quantiles")]
aux_parameters = [p for n, p in model.named_parameters() if n.endswith(".quantiles")]

optimizer = optim.Adam(parameters, lr=1e-4)
aux_optimizer = optim.Adam(aux_parameters, lr=1e-3)

# 3. Rate-Distortion 손실
mse_loss = nn.MSELoss()
lmbda = 0.0035

epochs = 3
print("🔥 2. 본격적인 학습을 시작합니다! (노이즈 제거 + 압축 동시 학습)\n")

for epoch in range(epochs):
    epoch_loss = 0.0
    for i, (noisy_img, clean_img) in enumerate(train_loader):
        noisy_img = noisy_img.to(device)
        clean_img = clean_img.to(device)

        optimizer.zero_grad()
        aux_optimizer.zero_grad()

        # [Forward] 노이즈 낀 사진을 입력으로
        out_net = model(noisy_img)
        x_hat = out_net['x_hat']
        likelihoods = out_net['likelihoods']

        # [Loss] BPP(압축량) + Distortion(복원 화질)
        N, _, H, W = noisy_img.size()
        num_pixels = N * H * W
        bpp_loss = sum(
            (torch.log(likelihoods[k]).sum() / (-math.log(2) * num_pixels))
            for k in likelihoods.keys()
        )
        # ★ 복원 결과 x_hat 을 '깨끗한 정답 clean_img'과 비교 (노이즈 제거 학습)
        mse = mse_loss(x_hat, clean_img)
        loss = bpp_loss + lmbda * (255 ** 2) * mse

        # [Backward]
        loss.backward()
        optimizer.step()

        aux_loss = model.aux_loss()
        aux_loss.backward()
        aux_optimizer.step()

        epoch_loss += loss.item()
        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Batch [{i}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f} | BPP: {bpp_loss.item():.4f} | MSE: {mse.item():.4f}")

    print(f"✅ Epoch [{epoch+1}/{epochs}] 완료! Avg Loss: {epoch_loss/len(train_loader):.4f}")
    print("-" * 50)

print("🎉 학습이 모두 완료되었습니다!")

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/CUAI_summer_conference/Baseline2.pth')
print("💾 Baseline 2(가우시안 JDC) 모델 가중치 저장 완료!")

In [ ]:
# PSNR 계산 함수 (화질 지표: 클수록 좋음)
def calc_psnr(a, b):
    mse = torch.mean((a - b) ** 2).item()
    if mse == 0:
        return 100.0
    return 10 * math.log10(1.0 / mse)

print("🪄 [테스트 A] 학습에 쓴 것과 같은 '인공 가우시안 노이즈'에서 성능 확인")
model.eval()

noisy_img, clean_img = next(iter(train_loader))
noisy_img = noisy_img.to(device)

with torch.no_grad():
    out_net = model(noisy_img)
    x_hat = out_net['x_hat'].clamp(0, 1)

noisy_cpu = noisy_img.cpu()
clean_cpu = clean_img.cpu()
x_hat_cpu = x_hat.cpu()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i in range(3):
    p_in = calc_psnr(noisy_cpu[i], clean_cpu[i])
    p_out = calc_psnr(x_hat_cpu[i], clean_cpu[i])

    axes[i, 0].imshow(noisy_cpu[i].permute(1, 2, 0))
    axes[i, 0].set_title(f"1. Noisy Input (PSNR {p_in:.2f})")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(x_hat_cpu[i].permute(1, 2, 0))
    axes[i, 1].set_title(f"2. Restored (PSNR {p_out:.2f})")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(clean_cpu[i].permute(1, 2, 0))
    axes[i, 2].set_title("3. Clean (GT)")
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()
print("👉 PSNR이 Noisy→Restored로 올라가면, 인공 노이즈는 잘 제거된 것입니다.")

## 🔬 [테스트 B] Domain Shift 검증 — 진짜 야간(LOL) 사진 넣어보기

이 모델은 **"밝은 사진 + 인공 가우시안 노이즈"** 만 배웠습니다.
그래서 **실제 저조도(어두운) LOL 사진**을 넣으면:
- 노이즈 패턴이 학습한 것과 다르고 (Real vs Gaussian)
- 애초에 '밝기 개선'은 배운 적이 없어서

→ **결과가 여전히 어둡고 노이즈/뭉개짐이 남는 실패**가 예상됩니다. 이게 연구 계획서에서 증명하려는 핵심입니다.

In [ ]:
print("🌙 [테스트 B] 학습한 적 없는 '진짜 야간(LOL low)' 사진으로 Domain Shift 확인")

# 실제 저조도 사진(low)과 정답(high)을 그대로 불러오는 로더 (노이즈 인공 주입 X)
class RealLOLPair(Dataset):
    def __init__(self, root_dir, crop_size=256):
        self.low_dir = os.path.join(root_dir, 'low')
        self.high_dir = os.path.join(root_dir, 'high')
        self.names = sorted(os.listdir(self.low_dir))
        self.crop_size = crop_size
    def __len__(self):
        return len(self.names)
    def __getitem__(self, idx):
        name = self.names[idx]
        low = Image.open(os.path.join(self.low_dir, name)).convert('RGB')
        high = Image.open(os.path.join(self.high_dir, name)).convert('RGB')
        w, h = TF.get_image_size(low)
        ci = random.randint(0, h - self.crop_size)
        cj = random.randint(0, w - self.crop_size)
        low = TF.crop(low, ci, cj, self.crop_size, self.crop_size)
        high = TF.crop(high, ci, cj, self.crop_size, self.crop_size)
        return TF.to_tensor(low), TF.to_tensor(high)

# eval15(평가셋)로 검증 — 학습에 안 쓴 데이터
eval_path = '/content/drive/MyDrive/CUAI_summer_conference/LOL_Dataset/lol_dataset/eval15'
real_dataset = RealLOLPair(root_dir=eval_path, crop_size=256)
real_loader = DataLoader(real_dataset, batch_size=3, shuffle=True)

low_img, high_img = next(iter(real_loader))
low_img = low_img.to(device)

with torch.no_grad():
    out_net = model(low_img)
    x_hat = out_net['x_hat'].clamp(0, 1)

low_cpu = low_img.cpu()
high_cpu = high_img.cpu()
x_hat_cpu = x_hat.cpu()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i in range(3):
    p_out = calc_psnr(x_hat_cpu[i], high_cpu[i])
    axes[i, 0].imshow(low_cpu[i].permute(1, 2, 0))
    axes[i, 0].set_title("1. Real Low-light (Input)")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(x_hat_cpu[i].permute(1, 2, 0))
    axes[i, 1].set_title(f"2. Model Output (PSNR {p_out:.2f})")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(high_cpu[i].permute(1, 2, 0))
    axes[i, 2].set_title("3. Bright GT")
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()
print("👉 예상대로 결과가 여전히 어둡고 개선이 안 되면 = 가우시안 JDC의 Domain Shift 실패 증명 ✅")